Different types of messages

System Messages

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [4]:


from langchain.chat_models import init_chat_model

model = init_chat_model("groq:openai/gpt-oss-120b")



In [11]:
from langchain_core.messages import SystemMessage,HumanMessage


messages = [
    SystemMessage("you are a science teacher"),
    HumanMessage("what is mass")
]
responce = model.invoke(messages)
print(responce.content)

**Mass – A Fundamental Property of Matter**

---

### 1. What is mass?

Mass is a measure of the amount of matter contained in an object.  
It tells us how much “stuff” makes up the object, independent of where the object is located or what forces act on it.

- **Intrinsic**: Mass does not change if you move the object from Earth to the Moon, or even into outer space.  
- **Scalar quantity**: It has magnitude but no direction (unlike a vector such as force).

---

### 2. How do we quantify mass?

| Quantity | Symbol | SI Unit | Typical Everyday Examples |
|----------|--------|--------|----------------------------|
| Mass     | *m*    | kilogram (kg) | A textbook ≈ 1 kg, a newborn baby ≈ 3 kg |
| Sub‑unit | —      | gram (g) = 10⁻³ kg | A paperclip ≈ 1 g |
| Large masses | —   | tonne (t) = 10³ kg | A car ≈ 1.5 t, a ship ≈ 10⁴ t |

Mass can also be expressed in **atomic mass units (u or amu)** for individual atoms and molecules, where 1 u ≈ 1.660 × 10⁻²⁷ kg.

---

### 3. Mass vs. Weight

In [14]:
human_msg = HumanMessage(
    content= "hello bot",
    name= "sendhaane",
    id = "msg1"
)


responce = model.invoke([
    SystemMessage("you are a professional bot"),
    human_msg
])

print(responce.content)
print(responce.usage_metadata)

Hello! How can I assist you today?
{'input_tokens': 81, 'output_tokens': 63, 'total_tokens': 144, 'output_token_details': {'reasoning': 45}}


In [ ]:
from langchain_core.messages import AIMessage
human_msg = HumanMessage(
    content= "hello bot, where is my room",
    name= "sendhaane",
    id = "msg2"
)

sys_msg = SystemMessage("you are a professional hotel manager")

ai_msg = AIMessage("hi how can i help you")
responce = model.invoke([
    sys_msg,
    ai_msg,
    
    human_msg
])

print(responce.content)
print(responce.usage_metadata)

Hello! I’m happy to help you locate your room.

Could you please provide a few details so I can find it quickly?

1. **Your name** (as it appears on the reservation)  
2. **Reservation number** (if you have it)  
3. **Floor or wing** you were assigned (if you know it)  

Once I have that information, I’ll let you know exactly where your room is and give you any directions you need to get there. If you’re already in the lobby, I can also arrange for a bellhop to escort you. Let me know!
{'input_tokens': 99, 'output_tokens': 171, 'total_tokens': 270, 'output_token_details': {'reasoning': 40}}


In [ ]:


from langchain_core.messages import AIMessage,ToolMessage
from langchain.tools import tool

@tool
def find_room (name:str)->int:
    """Find the room number assigned to a hotel guest using their name."""
    return 102



human_msg = HumanMessage(
    content= "Hello bot, where is my room? My name is sendhaane.",
    name= "sendhaane",
    id = "msg2"
)

model_with_tool = model.bind_tools([find_room])




sys_msg = SystemMessage("you are a professional hotel manager","Use the find_room tool to find guest room numbers.")


responce = model_with_tool.invoke([
    sys_msg,
    human_msg
])

print("AI tool call")
print(responce)
print(responce.usage_metadata)

print(responce.tool_calls)





AI tool call
content='' additional_kwargs={'reasoning_content': 'We need to use find_room tool with name "sendhaane".', 'tool_calls': [{'id': 'fc_7c438b82-a3e5-4875-8a81-1d760f46167d', 'function': {'arguments': '{"name":"sendhaane"}', 'name': 'find_room'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 155, 'total_tokens': 199, 'completion_time': 0.092041926, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.007243327, 'prompt_tokens_details': None, 'queue_time': 0.394124219, 'total_time': 0.099285253}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e1a78f200e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1a1ba-2625-702c-8bd7-5e7c82023cfc-0' tool_calls=[{'name': 'find_room', 'args': {'name': 'sendhaane'}, 'id': 'fc_7c438b82-a3e5-4875-8a81-1d760f46167d', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_toke

In [42]:
from langchain_core.messages import ToolMessage


tool_responce = responce.tool_calls[0]

tool_result = find_room.invoke(tool_responce["args"])

print("Tool result:", tool_result)


tool_msg = ToolMessage(
    content=str(tool_result),
    tool_call_id=tool_responce["id"]
)

final_responce = model_with_tool.invoke([
    sys_msg,
    human_msg,
    responce,
    tool_msg
])

print(final_responce.content)
print(final_responce.usage_metadata)



Tool result: 102
Hello Sendhaane! Your room number is **102**. If you need anything else—directions, hotel amenities, or assistance with anything—just let me know!
{'input_tokens': 183, 'output_tokens': 57, 'total_tokens': 240, 'output_token_details': {'reasoning': 13}}
